In [ ]:
## Nikolay Vorontsov,
## Mushroom task
## Inference with fine-tuned model

In [2]:
!pip install transformers huggingface_hub pip torch jsonlines regex
# Install necessary dependencies
!pip install transformers peft accelerate huggingface_hub
!pip install -q trl xformers wandb datasets einops sentencepiece
!pip install -U datasets bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.4/293.4 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 73.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 15.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.7/69.7 MB 9.3 MB/s eta 0:00:00


In [3]:
import jsonlines
import re
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification

from huggingface_hub import login


from google.colab import userdata

HUGGING_API = userdata.get('HUGGINGFACE_READ_AND_WRITE')

In [4]:

# Login to Hugging Face
login(token=HUGGING_API)


In [5]:
tokenizer = AutoTokenizer.from_pretrained("nicksnlp/llama-7B-hallucination")
model = AutoModelForTokenClassification.from_pretrained("nicksnlp/llama-7B-hallucination")

# Check for CUDA availability and set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
"""
def infer_with_model(input_text):
    inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
    predicted_labels = torch.argmax(logits, dim=-1)
    tokens = tokenizer.tokenize(input_text)
    labeled_tokens = list(zip(tokens, predicted_labels[0].tolist()))
    hallucinated_words = [token for token, label in labeled_tokens if label == 1]
    return hallucinated_words
"""
def infer_with_model(input_text):
    # Tokenize the input text
    inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True, max_length=128)

    # Move input tensors to the same device as the model
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # Predict the token labels (hallucination vs. correct)
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits  # Raw logits output from the model

    # Get the predicted labels (0 for correct, 1 for hallucinated)
    predicted_labels = torch.argmax(logits, dim=-1)

    # Decode the tokens from the input text
    tokens = tokenizer.tokenize(input_text)

    # Get the corresponding predicted labels for each token
    labeled_tokens = list(zip(tokens, predicted_labels[0].tolist()))

    # Create a list of hallucinated words
    hallucinated_words = [token for token, label in labeled_tokens if label == 1]

    return hallucinated_words


tokenizer_config.json:   0%|          | 0.00/978 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.62M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/437 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.22k [00:00<?, ?B/s]

Unused kwargs: ['_load_in_4bit', '_load_in_8bit', 'quant_method']. These kwargs are not used in <class 'transformers.utils.quantization_config.BitsAndBytesConfig'>.
`low_cpu_mem_usage` was None, now default to True since model is quantized.


model.safetensors:   0%|          | 0.00/3.91G [00:00<?, ?B/s]

In [6]:
# Example usage of the inference function
input_text = "Alexanderplatz is located in London City, it has been there since 1966."
hallucinated_words = infer_with_model(input_text)

# Print the list of hallucinated words
print("Hallucinated words:")
print(hallucinated_words)
print(list(tokenizer.decode(tokenizer.convert_tokens_to_ids(word)) for word in hallucinated_words))

Hallucinated words:
['platz', '▁located', '▁City', '▁it', '▁has', '▁been', '6', '.']
['platz', 'located', 'City', 'it', 'has', 'been', '6', '.']


In [7]:

def find_spans(model_output_text, hallucinated_words):
    spans = []
    for word in hallucinated_words:
        for match in re.finditer(re.escape(word), model_output_text):
            spans.append([match.start(), match.end()])
    return spans


In [8]:
find_spans(input_text, hallucinated_words)

[[9, 14], [68, 69], [69, 70], [70, 71]]

In [9]:

def group_adjacent_words(hallucinated_words): # "New" "New York" -> "New York"
    grouped_words = []
    i = 0
    while i < len(hallucinated_words):
        current_group = [hallucinated_words[i]]
        j = i + 1
        while j < len(hallucinated_words):
            if j < len(hallucinated_words) and hallucinated_words[j].startswith(hallucinated_words[i] + " "):
                current_group.append(hallucinated_words[j])
                i = j
            elif j < len(hallucinated_words) and hallucinated_words[i].startswith(hallucinated_words[j] + " "):
                current_group.insert(0, hallucinated_words[j])
                i = j
            else:
                break
            j += 1
        grouped_words.append(current_group[-1])
        i += 1
    return grouped_words


In [11]:
group_adjacent_words(list(tokenizer.decode(tokenizer.convert_tokens_to_ids(word)) for word in hallucinated_words))

['platz', 'located', 'City', 'it', 'has', 'been', '6', '.']

In [ ]:
"""
def process_validation_set(validation_file, output_file):
    with jsonlines.open(validation_file) as reader, jsonlines.open(output_file, 'w') as writer:
        for datapoint in reader:
            model_output_text = datapoint.get("model_output_text", "")
            input_text = datapoint.get("input_text", "")

            hallucinated_words = infer_with_model(input_text)

            decoded_hallucinated_words = [tokenizer.decode(tokenizer.convert_tokens_to_ids(word)).strip() for word in hallucinated_words]
            decoded_hallucinated_words = list(filter(None, decoded_hallucinated_words))

            grouped_hallucinated_words = group_adjacent_words(decoded_hallucinated_words)

            spans = find_spans(model_output_text, grouped_hallucinated_words)
            datapoint["hallucinated_spans"] = spans
            writer.write(datapoint)
"""

In [29]:
## Merge overlapping spans

def merge_hallucinated_spans(spans):
    """
    Unites overlapping or adjacent spans into single ranges.
    """
    # Sort the spans by the starting index
    spans.sort(key=lambda x: x[0])

    # Initialize the merged spans list
    merged_spans = []

    for span in spans:
        # If the merged list is empty or the current span does not overlap with the last merged span
        if not merged_spans or merged_spans[-1][1] < span[0] - 1:
            merged_spans.append(span)  # Add as a new non-overlapping span
        else:
            # Otherwise, merge with the last span
            merged_spans[-1][1] = max(merged_spans[-1][1], span[1])

    return merged_spans


In [25]:
def test_inferences(validation_file, output_file):
  with jsonlines.open(validation_file) as reader, jsonlines.open(output_file, 'w') as writer:
    for datapoint in reader:
            model_output_text = datapoint.get("model_output_text", "")
            input_text = datapoint.get("model_input", "")

            hallucinated_words = infer_with_model(input_text+model_output_text)

            decoded_hallucinated_words = [tokenizer.decode(tokenizer.convert_tokens_to_ids(word)).strip() for word in hallucinated_words]
            decoded_hallucinated_words = list(filter(None, decoded_hallucinated_words))

            grouped_hallucinated_words = group_adjacent_words(decoded_hallucinated_words)

            spans = find_spans(model_output_text, grouped_hallucinated_words)

            datapoint["hallucinated_spans"] = merge_hallucinated_spans(spans)
            print(datapoint["hallucinated_spans"])
            datapoint["hallucinated_words"] = grouped_hallucinated_words
            print(datapoint["hallucinated_words"])

            #datapoint["id"] = datapoint["id"].replace('_unlabeled', '')

            writer.write(datapoint)

In [27]:
# "a" creates the file if it doesn't exist
with open(output_file, "a") as file:
    pass  # Do nothing, just ensure the file exists

print(f"{output_file} is created or already exists.")

/content/validation_with_spans.jsonl is created or already exists.


In [28]:

# Example usage:
validation_file = "/content/mushroom.en-val.v2.unlabeled.jsonl"
output_file = "/content/validation_with_spans.jsonl"
#process_validation_set(validation_file, output_file)
test_inferences(validation_file, output_file)
print(f"Processed data written to {output_file}")

[[16, 18], [4, 5], [7, 8], [23, 24], [35, 36], [82, 83], [32, 37], [46, 47], [47, 48], [46, 47], [47, 48], [48, 49], [71, 73], [73, 76]]
['Sta', 'en', 'win', 'a', 'medal', 'for', '?', '0', '0', '8', 'ij', 'ing']
[[11, 12], [25, 26], [37, 38], [7, 9], [21, 28], [7, 9], [11, 14], [15, 20], [21, 29], [32, 37], [11, 12], [25, 26], [37, 38]]
['many', 'a', 'does', 'si', 'ales', 'contain', 'si', 'ale', 'order', 'contains', 'gener', 'a']
[[5, 8], [47, 50], [54, 56], [74, 76], [19, 23], [24, 29], [29, 31], [78, 80], [1, 2], [22, 23], [27, 28], [37, 38], [39, 40], [56, 57], [61, 62], [69, 70], [76, 77], [81, 82], [84, 85], [86, 87], [5, 8], [47, 50], [9, 12], [12, 15], [15, 18], [19, 23], [24, 29], [29, 32], [32, 33], [87, 88], [3, 4], [41, 42], [43, 46], [5, 8], [47, 50], [51, 53], [54, 58], [59, 62], [63, 70], [71, 73], [14, 15], [25, 26], [28, 29], [29, 30], [43, 44], [78, 79], [79, 83], [84, 87], [32, 33], [87, 88]]
['all', 'th', 'rop', 'ods', 'have', 'anten', 'na', 'e', '?', 'all', 'ara', '

In [22]:
test_inferences("/content/mushroom.en-val.v2.unlabeled.jsonl")

[[16, 18], [4, 5], [7, 8], [23, 24], [35, 36], [82, 83], [32, 37], [46, 47], [47, 48], [46, 47], [47, 48], [48, 49], [71, 73], [73, 76]]
['Sta', 'en', 'win', 'a', 'medal', 'for', '?', '0', '0', '8', 'ij', 'ing']
[[11, 12], [25, 26], [37, 38], [7, 9], [21, 28], [7, 9], [11, 14], [15, 20], [21, 29], [32, 37], [11, 12], [25, 26], [37, 38]]
['many', 'a', 'does', 'si', 'ales', 'contain', 'si', 'ale', 'order', 'contains', 'gener', 'a']
[[5, 8], [47, 50], [54, 56], [74, 76], [19, 23], [24, 29], [29, 31], [78, 80], [1, 2], [22, 23], [27, 28], [37, 38], [39, 40], [56, 57], [61, 62], [69, 70], [76, 77], [81, 82], [84, 85], [86, 87], [5, 8], [47, 50], [9, 12], [12, 15], [15, 18], [19, 23], [24, 29], [29, 32], [32, 33], [87, 88], [3, 4], [41, 42], [43, 46], [5, 8], [47, 50], [51, 53], [54, 58], [59, 62], [63, 70], [71, 73], [14, 15], [25, 26], [28, 29], [29, 30], [43, 44], [78, 79], [79, 83], [84, 87], [32, 33], [87, 88]]
['all', 'th', 'rop', 'ods', 'have', 'anten', 'na', 'e', '?', 'all', 'ara', '

In [31]:
import json

In [32]:
# Clean_output

with open(output_file, "r", encoding='utf-8') as jsonl_file:
    lines = jsonl_file.readlines()

    for line in lines:

        # Remove '_unlabeled' from the 'id' field
        data_to_resave = json.loads(line)

        data_to_resave['id'] = data_to_resave['id'].replace('_unlabeled', '')
        print(data_to_resave['id'])
        data_to_resave["hard_labels"] = unite_hallucinated_spans(data_to_resave["hallucinated_spans"])
        print(data_to_resave["hard_labels"])

        soft_labels = [{'start': label[0], 'prob': float(1), 'end': label[1]} for label in data_to_resave['hard_labels'] if label]
        print(soft_labels)


             # Save the datapoint to the JSONL file
        with open(f"{output_file}_no_extra_keys_soft_labels_prob1.jsonl", "a", encoding='utf-8') as jsonl_file:
            datapoint_labelled = {
                "id":data_to_resave["id"],
                "lang":data_to_resave["lang"],
                "model_input":data_to_resave["model_input"],
                "model_output_text":data_to_resave["model_output_text"],
                "model_id":data_to_resave["model_id"],
                "soft_labels":soft_labels, #instead of data_to_resave["soft_labels"], that is to output an empty list.
                "hard_labels":data_to_resave["hard_labels"],
                "model_output_logits":data_to_resave["model_output_logits"],
                "model_output_tokens":data_to_resave["model_output_tokens"],
            }
            jsonl_file.write(json.dumps(datapoint_labelled) + "\n")

val-en-1
[[4, 5], [7, 8], [16, 18], [23, 24], [32, 37], [46, 49], [71, 76], [82, 83]]
[{'start': 4, 'prob': 1.0, 'end': 5}, {'start': 7, 'prob': 1.0, 'end': 8}, {'start': 16, 'prob': 1.0, 'end': 18}, {'start': 23, 'prob': 1.0, 'end': 24}, {'start': 32, 'prob': 1.0, 'end': 37}, {'start': 46, 'prob': 1.0, 'end': 49}, {'start': 71, 'prob': 1.0, 'end': 76}, {'start': 82, 'prob': 1.0, 'end': 83}]
val-en-2
[[7, 9], [11, 29], [32, 38]]
[{'start': 7, 'prob': 1.0, 'end': 9}, {'start': 11, 'prob': 1.0, 'end': 29}, {'start': 32, 'prob': 1.0, 'end': 38}]
val-en-3
[[1, 33], [37, 88]]
[{'start': 1, 'prob': 1.0, 'end': 33}, {'start': 37, 'prob': 1.0, 'end': 88}]
val-en-4
[[2, 10], [13, 28], [33, 34]]
[{'start': 2, 'prob': 1.0, 'end': 10}, {'start': 13, 'prob': 1.0, 'end': 28}, {'start': 33, 'prob': 1.0, 'end': 34}]
val-en-5
[[10, 20], [22, 25], [31, 36], [47, 51], [55, 63], [66, 67], [70, 71], [79, 83], [92, 111], [115, 121], [123, 137], [141, 154], [175, 189], [198, 215], [223, 228]]
[{'start': 10, 